# 02 — Pipeline TDA con datos reales (CBIS-DDSM)

Entorno de trabajo interactivo sobre el pipeline real (no sintético). Reutiliza
exactamente el mismo código que `scripts/run_pipeline.py`, `scripts/train_cnn.py`
y `scripts/fuse_and_evaluate.py` — este notebook es para explorar/depurar celda
a celda, los scripts son para reproducir resultados de un tirón.

**Requisito previo:** haber descargado los CSV reales (`data/csv/`) y al menos
un piloto de imágenes (`python scripts/download_cbis_ddsm.py --add 15 15`).


## 0. Configuración e imports

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

from src.data import load_case_descriptions, split_by_patient
from src.preprocessing import preprocess
from src.topology import backend, persistence_diagram, persistence_image_vector
from src.evaluate import clinical_metrics, plot_confusion, plot_persistence_diagram
from src.models import build_random_forest, build_svm

import run_pipeline as rp  # funciones compartidas: _load_case_images, build_dataset

print("Backend TDA activo:", backend())


## 1. Metadatos reales

Carga los 4 CSV de CBIS-DDSM (con la etiqueta benigno/maligno real) y particiona
por paciente (nunca por imagen, para no repetir la fuga de datos del
`Demo_Notebook.ipynb` original).

In [ ]:
df = load_case_descriptions(ROOT / "data" / "csv")
df_mass = df[df["_prefix"].str.startswith("Mass")].reset_index(drop=True)
print(f"{len(df_mass)} filas de casos de masa, {df_mass['patient_id'].nunique()} pacientes únicos")
df_mass["pathology"].value_counts()


In [ ]:
partitions = split_by_patient(df_mass)
for name, part in partitions.items():
    print(f"{name}: {len(part)} filas, {part['patient_id'].nunique()} pacientes")


## 2. Cargar y previsualizar un caso real

`rp._load_case_images(row)` ya sabe reconstruir el nombre de carpeta real de TCIA
y distinguir la máscara ROI del recorte pre-hecho (ver nota en
`scripts/download_cbis_ddsm.py:download_case`). Devuelve `None` si ese caso
concreto aún no se ha descargado.

In [ ]:
ejemplo = None
for _, row in df_mass.iterrows():
    loaded = rp._load_case_images(row)
    if loaded is not None:
        ejemplo = (row, *loaded)
        break

if ejemplo is None:
    print("Ningún caso descargado todavía — amplía el piloto con scripts/download_cbis_ddsm.py")
else:
    row, full_image, mask = ejemplo
    print(f"Caso: {row['patient_id']} ({row['left_or_right_breast']} {row['image_view']}), "
          f"pathology={row['pathology']}")

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    axes[0].imshow(full_image, cmap="gray"); axes[0].set_title("Mamografía completa"); axes[0].axis("off")
    if mask is not None:
        axes[1].imshow(mask, cmap="gray"); axes[1].set_title("Máscara ROI")
    else:
        axes[1].text(0.5, 0.5, "sin máscara\n(caso raro, ver nota)", ha="center")
    axes[1].axis("off")
    plt.show()


## 3. Preprocesado + descriptor topológico de ese mismo caso

In [ ]:
if ejemplo is not None:
    img_pp = preprocess(full_image, mask=mask)
    diagrama = persistence_diagram(img_pp)
    vector = persistence_image_vector(diagrama, n_bins=4)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(img_pp, cmap="gray"); axes[0].set_title("Preprocesada (224x224)"); axes[0].axis("off")
    plot_persistence_diagram(diagrama, ax=axes[1])
    plt.tight_layout(); plt.show()

    print("Vector de features (n_bins=4):", vector.shape)


## 4. Dataset completo (todos los casos ya descargados)

Usa `rp.build_dataset`, la misma función que `scripts/run_pipeline.py`: recorre
el DataFrame, carga cada imagen disponible, preprocesa y extrae el vector
topológico. Los casos sin imagen descargada se omiten (y se cuentan).

In [ ]:
X_train, y_train, train_rows = rp.build_dataset(partitions["train"], n_bins=4)
X_test, y_test, test_rows = rp.build_dataset(partitions["test"], n_bins=4)

print(f"Usables: {len(X_train)} train, {len(X_test)} test")
print("Balance train:", np.bincount(y_train), " | Balance test:", np.bincount(y_test))


## 5. Clasificador topológico + métricas clínicas

In [ ]:
if len(X_train) and len(X_test):
    clf = build_random_forest()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_score = clf.predict_proba(X_test)[:, 1]

    metrics = clinical_metrics(y_test, y_pred, y_score=y_score)
    for k, v in metrics.items():
        print(f"{k}: {v:.3f}")

    plot_confusion(y_test, y_pred)
    plt.show()
else:
    print("Amplía el piloto antes de entrenar: python scripts/download_cbis_ddsm.py --add 15 15")


## 6. Validación por CV (más fiable que el test único)

Con pilotos pequeños, el AUC de un solo split de test es ruidoso (se observó
variar entre 0.4 y 0.65 solo por el tamaño de muestra). La media de validación
cruzada sobre train es una estimación más estable.

In [ ]:
from sklearn.model_selection import cross_val_score

if len(X_train) >= 10:
    cv_auc = cross_val_score(build_random_forest(), X_train, y_train, cv=5, scoring="roc_auc")
    print(f"AUC (5-fold CV sobre train): {cv_auc.mean():.3f} +/- {cv_auc.std():.3f}")


## 7. CNN de referencia + fusión (opcional, más lento)

Reutiliza `scripts/train_cnn.py` (PyTorch, DenseNet121, semilla fija). Puede
tardar unos minutos en CPU.

In [ ]:
import train_cnn as tc

X_img_train, y_img_train = tc.build_image_dataset(partitions["train"])
X_img_test, y_img_test = tc.build_image_dataset(partitions["test"])

if len(X_img_train):
    model, device = tc.train_cnn(X_img_train, y_img_train, epochs=10, lr=1e-4, batch_size=8)
    y_pred_cnn, y_score_cnn = tc.evaluate_cnn(model, device, X_img_test, y_img_test)
    print(clinical_metrics(y_img_test, y_pred_cnn, y_score=y_score_cnn))


## 8. Caso de estudio: ¿depende el resultado de la función de filtración?

Toda la homología persistente de arriba usa la **intensidad** del píxel como
"altura". Una alternativa es la **distancia al fondo** (transformada de
distancia: sensible a la forma y grosor de las regiones más que a su brillo).
Si el clasificador solo funcionara con una de las dos, el resultado sería un
artefacto de esa elección (issue #77).

In [ ]:
from sklearn.model_selection import cross_val_score

auc_filtracion = {}
for filtracion in ["intensity", "distance_transform"]:
    X_f, y_f, _ = rp.build_dataset(partitions["train"], n_bins=4, filtration=filtracion)
    s = cross_val_score(build_random_forest(), X_f, y_f, cv=5, scoring="roc_auc")
    auc_filtracion[filtracion] = (s.mean(), s.std())
    print(f"{filtracion:20s}: AUC (5-fold CV) = {s.mean():.3f} +/- {s.std():.3f}")

fig, ax = plt.subplots(figsize=(5, 3))
nombres = list(auc_filtracion)
ax.bar(nombres, [auc_filtracion[n][0] for n in nombres],
       yerr=[auc_filtracion[n][1] for n in nombres], capsize=4, color=["#1d5a99", "#5a991d"])
ax.axhline(0.5, color="k", ls="--", lw=0.8); ax.set_ylim(0, 1); ax.set_ylabel("AUC (5-fold CV)")
plt.tight_layout(); plt.show()


## 9. Caso de estudio: ¿funciona igual en todos los subgrupos clínicos?

Un AUC global puede esconder que el clasificador topológico solo funciona en
un tipo de paciente. Con predicciones fuera de muestra (validación cruzada)
calculamos el AUC por **densidad mamaria** (BI-RADS 1-4: las mamas densas son
las más difíciles de leer) y por **margen de la masa** (espiculado,
circunscrito...). Solo se reportan subgrupos con ≥10 casos y ambas clases.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict

prob = cross_val_predict(build_random_forest(), X_train, y_train, cv=5, method="predict_proba")[:, 1]
filas = pd.DataFrame(train_rows).reset_index(drop=True)
filas["prob"], filas["y"] = prob, y_train

for campo in ["breast_density", "mass_margins"]:
    print()
    print(f"AUC por {campo}:")
    for valor, grupo in filas.groupby(campo):
        if len(grupo) < 10 or grupo["y"].nunique() < 2:
            print(f"  {str(valor):28s} n={len(grupo):3d}  (insuficiente)")
            continue
        print(f"  {str(valor):28s} n={len(grupo):3d}  AUC={roc_auc_score(grupo['y'], grupo['prob']):.3f}")


## Siguientes pasos

- Ampliar el piloto: `python scripts/download_cbis_ddsm.py --add 15 15`
- La hipótesis principal del TFG (detección anticipada longitudinal) vive en
  **`03_embed_longitudinal.ipynb`**, sobre datos reales de EMBED.
- Estado de la investigación y decisiones: issues de GitHub.
